# Section 1 · Your First AI Agent (Groq edition)

In this notebook you build your first AI agent with the [Strands Agents SDK](https://strandsagents.com), powered by [Groq](https://console.groq.com) instead of Amazon Bedrock. Only the model setup (Steps 1–3) differs from `01_hello_agent.ipynb`; the agent code is identical.

You will:
1. Install the SDK
2. Pick a Groq model
3. Create an agent in one line
4. Give it your own tool with `@tool`
5. Give it a prebuilt tool
6. Shape its behaviour with a system prompt

**Before you start**
- A Groq API key from [console.groq.com/keys](https://console.groq.com/keys)
- Set it as `GROQ_API_KEY` in your environment or in a `.env` file (this folder or the repo root)

## Step 1: Install dependencies

In [ ]:
#%pip install -q -r requirements-groq.txt

Note: you may need to restart the kernel to use updated packages.


## Step 2: Choose the model

Every example in this course reads the model from one variable. Change it here (or set the `GROQ_MODEL_ID` environment variable) to try a different model.

- `openai/gpt-oss-120b` — default, strong reasoning and tool use
- `openai/gpt-oss-20b` — faster and cheaper
- `qwen/qwen3.8-27b` — an alternative open-weight model

See [console.groq.com/docs/models](https://console.groq.com/docs/models) for the current list.

In [5]:
import os

from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

print(f"Model  : {MODEL_ID}")
print(f"API key: {'set' if os.getenv('GROQ_API_KEY') else 'MISSING - set GROQ_API_KEY'}")

Model  : openai/gpt-oss-120b
API key: MISSING - set GROQ_API_KEY


## Step 3: Create your first agent

Groq exposes an OpenAI-compatible API, so Strands' `OpenAIModel` talks to it by pointing `base_url` at Groq. Swapping `BedrockModel` for this is the only change: everything after this cell is the same as the Bedrock notebook.

In [3]:
from strands import Agent
from strands.models.openai import OpenAIModel

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)

# An agent = a model + (optional) tools + (optional) system prompt
agent = Agent(model=model)

response = agent("Explain Amazon Bedrock AgentCore in three sentences.")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

The agent streams its answer as it generates it. The returned object holds the full result — the final text, token usage, and the stop reason:

In [ ]:
print("Stop reason:", response.stop_reason)
print("Tokens used:", response.metrics.accumulated_usage)

## Step 4: Give the agent a tool

A tool is just a Python function with:
- the `@tool` decorator
- type hints (so the model knows what arguments to send)
- a docstring (so the model knows *when* to use it)

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

from strands import tool


@tool
def current_time(timezone: str = "Asia/Kolkata") -> str:
    """Get the current date and time in a given IANA timezone.

    Args:
        timezone: IANA timezone name, for example "Asia/Kolkata" or "America/New_York".
    """
    return datetime.now(ZoneInfo(timezone)).strftime("%A, %d %B %Y %I:%M %p %Z")


agent = Agent(model=model, tools=[current_time])
agent("What time is it right now in Bangalore and in New York?")

## Step 5: Use a prebuilt tool

`strands-agents-tools` ships ready-made tools. Here the agent gets a calculator so it does not have to do arithmetic "in its head".

In [ ]:
from strands_tools import calculator

agent = Agent(model=model, tools=[calculator, current_time])
agent("What is 18% GST on a laptop that costs Rs 84,999? Give me the final price too.")

## Step 6: Shape behaviour with a system prompt

The system prompt sets the agent's role, tone, and rules for every turn.

In [ ]:
agent = Agent(
    model=model,
    system_prompt=(
        "You are a patient tutor for engineers who are new to AI agents. "
        "Explain ideas with one real-world analogy and keep answers under 120 words."
    ),
)
agent("What is the difference between an LLM and an AI agent?")

## What you just did

- Built an agent with one line of code
- Added your own tool and a prebuilt tool
- Controlled behaviour with a system prompt
- Swapped the model provider from Bedrock to Groq without touching the agent code

Next, run the two Groq scripts in this folder to see the same idea as plain Python files, including one built with LangGraph:

```bash
python hello_agent_groq.py
python hello_langgraph_agent_groq.py
```